# §11.5.4 — 연산량 감소와 벽시계 시간 감소의 불일치

> 딥러닝 교재 · 3부 11장 5절 4항 (🐍)
> 선행: §11.5.1(감소 비율 $1/C+1/k^2$) · §11.5.3(손계산) · §11.4.3(MAC 세는 규칙)

## 이 노트북이 답하는 질문

1. **깊이별 분리의 MAC 절감이 시간 절감으로 그대로 이어지는가?** — 아니라는 것을 실측한다.
2. **왜 아닌가?** 연산 강도(FLOP/바이트)의 차이를 계산해 대조한다.
3. **MAC이 같아도 시간이 다를 수 있는가?** 같은 FLOP를 다른 모양으로 실행해 본다.

**예상 실행 시간** CPU 약 60초.
⚠︎ 시간 측정은 하드웨어·BLAS 구현에 따라 절대값이 크게 달라진다. **경향만** 보고,
절대 수치는 각자의 기계에서 다시 재야 한다 (§11.9.6의 규약).

---
## 0. 설정

In [ ]:
import os, glob, time
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

_t0 = time.time()

# ── 손잡이 (마지막 셀에 전체 목록) ──────────────────────────
FAST     = False     # True면 시행 수를 줄여 빠르게
SEED     = 20260808
SAVE_PDF = False     # True면 figs/에 교재용 PDF 저장
FIG_DIR  = 'figs'
# ──────────────────────────────────────────────────────────

# 색맹 안전 팔레트 (Okabe–Ito) — 규약 §II.4-7
CB = ['#000000', '#E69F00', '#56B4E9', '#009E73',
      '#D55E00', '#0072B2', '#CC79A7', '#F0E442']
plt.rcParams.update({'figure.dpi': 120, 'font.size': 10, 'axes.grid': True,
                     'grid.alpha': 0.3, 'axes.prop_cycle': plt.cycler(color=CB),
                     'figure.autolayout': True})

# 한글 폰트 (부록 K). 없으면 그림 라벨만 영문으로 대체한다.
for _p in glob.glob('/usr/share/fonts/**/*CJK*.ttc', recursive=True)[:6]:
    try:
        fm.fontManager.addfont(_p)
    except Exception:
        pass
_av = {f.name for f in fm.fontManager.ttflist}
KO_FONT = next((f for f in ['NanumGothic', 'Malgun Gothic', 'AppleGothic',
                            'Noto Sans CJK KR', 'Noto Sans KR', 'NanumBarunGothic',
                            'Noto Sans CJK JP'] if f in _av), None)
if KO_FONT:
    plt.rcParams['font.family'] = KO_FONT
    plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['pdf.fonttype'] = 42
plt.rcParams['ps.fonttype'] = 42
def lab(ko, en):
    return ko if KO_FONT else en

def save_book_fig(fig, name):
    # 교재 결합 그림 저장 — 불필요한 여백 없이
    if SAVE_PDF:
        os.makedirs(FIG_DIR, exist_ok=True)
        fig.savefig(os.path.join(FIG_DIR, name + '.pdf'),
                    bbox_inches='tight', pad_inches=0.03)
        print('저장:', os.path.join(FIG_DIR, name + '.pdf'))

rng = np.random.default_rng(SEED)
print(f"numpy {np.__version__}  |  FAST={FAST}  |  seed={SEED}")
print(f"한글 폰트: {KO_FONT or '없음 → 그림 라벨은 영문으로 출력됩니다'}")

---
## 1. 두 합성곱의 im2col 구현

표준 합성곱: im2col 후 **큰 행렬 곱 하나**. 깊이별 분리: 채널마다 **작은 곱** + $1\times1$의 곱.
같은 라이브러리(NumPy/BLAS) 위에서 모양만 다르게 실행된다.

In [ ]:
from numpy.lib.stride_tricks import sliding_window_view

def std_conv(X, W):
    # X: (N,C,H,Wd), W: (F, C*9) -> (N,F,Ho,Wo)
    N, C, H, Wd = X.shape
    v = sliding_window_view(X, (3, 3), axis=(2, 3))
    col = np.ascontiguousarray(v.transpose(0, 2, 3, 1, 4, 5)).reshape(N*(H-2)*(Wd-2), C*9)
    y = col @ W.T
    return y.reshape(N, H-2, Wd-2, -1).transpose(0, 3, 1, 2)

def dw_sep_conv(X, Wd_, Wp):
    # 깊이별: 채널별 (9,) 커널, 점별: (F, C)
    N, C, H, W2 = X.shape
    v = sliding_window_view(X, (3, 3), axis=(2, 3))          # (N,C,Ho,Wo,3,3)
    col = v.reshape(N, C, H-2, W2-2, 9)
    dwo = np.einsum('nchwk,ck->nchw', col, Wd_, optimize=True)
    y = np.einsum('nchw,fc->nfhw', dwo, Wp, optimize=True)
    return y

def bench(fn, *args, rep=5):
    fn(*args)                                   # 워밍업
    ts = []
    for _ in range(rep):
        t = time.perf_counter(); fn(*args); ts.append(time.perf_counter() - t)
    return min(ts)

# 정합성 확인: 분리 = (깊이별 후 점별)을 표준 합성곱 두 번으로도 계산해 일치 확인
Xt = rng.standard_normal((2, 8, 16, 16))
Wdw = rng.standard_normal((8, 9)); Wpp = rng.standard_normal((16, 8))
y1 = dw_sep_conv(Xt, Wdw, Wpp)
Wstd = np.zeros((8, 8*9))
for c in range(8):
    Wstd[c, c*9:(c+1)*9] = Wdw[c]
y2 = np.einsum('nchw,fc->nfhw', std_conv(Xt, Wstd), Wpp, optimize=True)
print("정합성:", np.allclose(y1, y2))

---
## 2. 채널 수를 훑는다 — MAC 비 vs 시간 비

In [ ]:
HH = 32; N_B = 4
CS = [16, 32, 64, 128] if FAST else [16, 32, 64, 128, 256]
mac_ratio, time_ratio, t_std_list, t_sep_list = [], [], [], []
for C in CS:
    F = C
    X = rng.standard_normal((N_B, C, HH, HH)).astype(np.float64)
    Wstd = rng.standard_normal((F, C*9))
    Wdw = rng.standard_normal((C, 9)); Wpp = rng.standard_normal((F, C))
    t_std = bench(std_conv, X, Wstd)
    t_sep = bench(dw_sep_conv, X, Wdw, Wpp)
    Ho = HH-2
    mac_std = N_B*Ho*Ho*C*F*9
    mac_sep = N_B*Ho*Ho*(C*9 + C*F)
    mac_ratio.append(mac_sep/mac_std); time_ratio.append(t_sep/t_std)
    t_std_list.append(t_std); t_sep_list.append(t_sep)
    print(f"C={C:3d}  MAC비={mac_sep/mac_std:.3f}  시간비={t_sep/t_std:.3f}  "
          f"(표준 {t_std*1e3:.1f}ms, 분리 {t_sep*1e3:.1f}ms)")

---
## 3. 연산 강도 — 산술 한 번에 몇 바이트를 옮기는가

FLOP 수를 (입력+가중치+출력을 한 번씩 읽고 쓴다고 가정한) 바이트 수로 나눈 **연산 강도**를 계산한다.
이 값이 낮으면 계산기가 아니라 메모리 대역폭이 병목이다 (§53.3 루프라인의 예고).

In [ ]:
def intensity(C, F, H, k=3, dw=False, dtype_bytes=8):
    Ho = H-2
    if not dw:
        flops = 2*Ho*Ho*C*F*k*k
        bytes_ = dtype_bytes*(C*H*H + F*C*k*k + F*Ho*Ho)
    else:
        flops = 2*Ho*Ho*C*k*k
        bytes_ = dtype_bytes*(C*H*H + C*k*k + C*Ho*Ho)
    return flops/bytes_

int_std = [intensity(C, C, HH) for C in CS]
int_dw  = [intensity(C, C, HH, dw=True) for C in CS]
for C, a, b in zip(CS, int_std, int_dw):
    print(f"C={C:3d}  표준 {a:6.1f} FLOP/B   깊이별 {b:5.2f} FLOP/B")

---
## 4. 같은 FLOP, 다른 모양

행렬 곱 $A\in\bbR^{1024\times1024}$ 하나와, 같은 총 FLOP인 $64\times64$ 곱 $4096$개를 비교한다.

In [ ]:
M = 1024; m = 64; K = (M//m)**3
A = rng.standard_normal((M, M)); B = rng.standard_normal((M, M))
a_s = rng.standard_normal((K, m, m)); b_s = rng.standard_normal((K, m, m))

def big():
    return A @ B
def small_loop():
    out = []
    for i in range(K):
        out.append(a_s[i] @ b_s[i])
    return out

t_big = bench(big, rep=3); t_small = bench(small_loop, rep=3)
print(f"같은 {2*M**3/1e9:.1f} GFLOP:  큰 곱 하나 {t_big*1e3:.0f}ms  vs  작은 곱 {K}개 {t_small*1e3:.0f}ms  ({t_small/t_big:.1f}배)")

---
## 5. 교재 그림 — fig_11_5_4

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(10.8, 6.9))
axes = axes.ravel()

# (a) MAC비 vs 시간비
ax = axes[0]
ax.plot(CS, mac_ratio, 'o-', color=CB[0], ms=4, label=lab('MAC 비 (분리/표준) $=1/C+1/9$', 'MAC ratio'))
ax.plot(CS, time_ratio, 's-', color=CB[4], ms=4, label=lab('실측 시간 비', 'measured time ratio'))
ax.set_xscale('log', base=2)
ax.axhline(1.0, color='k', lw=0.7, ls='--')
ax.text(CS[0], 1.05, lab('1.0 = 같은 속도', '1.0 = same speed'), fontsize=8)
ax.set_xlabel(lab('채널 수 $C$', 'channels $C$'))
ax.set_ylabel(lab('비율 (분리/표준)', 'ratio (separable/standard)'))
ax.set_title(lab('(a) 이론 절감과 실측 절감의 간극', '(a) MAC vs wall-clock savings'), fontsize=10)
ax.legend(fontsize=8)

# (b) 연산 강도
ax = axes[1]
w = 0.35; xs = np.arange(len(CS))
ax.bar(xs - w/2, int_std, w, color=CB[5], label=lab('표준 합성곱', 'standard'))
ax.bar(xs + w/2, int_dw, w, color=CB[1], label=lab('깊이별 합성곱', 'depthwise'))
ax.set_xticks(xs); ax.set_xticklabels(CS)
ax.set_yscale('log')
ax.set_xlabel(lab('채널 수 $C$', 'channels $C$'))
ax.set_ylabel(lab('연산 강도 (FLOP/바이트)', 'arithmetic intensity (FLOP/B)'))
ax.set_title(lab('(b) 깊이별 합성곱은 연산 강도가 낮다', '(b) arithmetic intensity'), fontsize=10)
ax.legend(fontsize=8)

# (c) 같은 FLOP 다른 모양
ax = axes[2]
bars = ax.bar([0, 1], [t_big*1e3, t_small*1e3], color=[CB[5], CB[4]], width=0.5)
ax.set_xticks([0, 1])
ax.set_xticklabels([lab('큰 곱 1개\n$1024^3$', 'one big GEMM'),
                    lab(f'작은 곱 {K}개\n$64^3\\times{K}$', f'{K} small GEMMs')], fontsize=9)
ax.set_ylabel(lab('시간 (ms)', 'time (ms)'))
for i, v in enumerate([t_big*1e3, t_small*1e3]):
    ax.text(i, v*1.02, f'{v:.0f}ms', ha='center', fontsize=9)
ax.set_title(lab('(c) 같은 FLOP, 다른 시간', '(c) same FLOPs, different time'), fontsize=10)

# (d) 절대 시간
ax = axes[3]
ax.plot(CS, np.array(t_std_list)*1e3, 'o-', color=CB[5], ms=4, label=lab('표준', 'standard'))
ax.plot(CS, np.array(t_sep_list)*1e3, 's-', color=CB[1], ms=4, label=lab('분리', 'separable'))
ax.set_xscale('log', base=2); ax.set_yscale('log')
ax.set_xlabel(lab('채널 수 $C$', 'channels $C$'))
ax.set_ylabel(lab('시간 (ms)', 'time (ms)'))
ax.set_title(lab('(d) 절대 실행 시간', '(d) absolute wall-clock'), fontsize=10)
ax.legend(fontsize=8)

save_book_fig(fig, 'fig_11_5_4')
plt.show()

> ### 읽는 법
>
> (a) MAC 비는 $C$가 커질수록 $1/9$ 근처로 내려가는데, **이 구현의 실측 시간 비는 1을 넘는다.**
> 8배 절감이라던 연산이 오히려 더 느리다. (b)가 원인이다. 깊이별 합성곱은 산술 한 번당
> 옮겨야 하는 바이트가 수십 배 많아(연산 강도 ≈ 1 FLOP/B) 메모리가 병목이 되고,
> BLAS의 빽빽한 행렬 곱 최적화도 받지 못한다. 전용 커널이 있는 프레임워크에서는 1 아래로
> 내려가지만 **MAC 비만큼 내려가는 일은 없다.** (c) 같은 FLOP도 실행 모양에 따라 몇 배 갈린다.
> **FLOPs는 속도의 대리 지표일 뿐이다** (§11.5.5). 이론적 틀은 §53.3 루프라인에서.

---
## 6. 자기 점검

1. (a)에서 $C$가 작을 때 간극이 더 큰 이유는? (힌트: 고정 비용과 행렬 크기)
2. (b)의 연산 강도 공식에서 캐시 재사용을 무시했다. 실제 강도는 계산값보다 높은가 낮은가?
3. GPU에서 이 실험을 반복하면 (a)의 간극은 커지겠는가 작아지겠는가? 왜인가?
4. §11.5.1의 비 $1/C_\text{out}+1/k^2$에서, $k=7$이면 절감의 상한은 얼마인가? ConvNeXt가 큰 커널을 깊이별로만 쓰는 이유와 연결하라.

## 7. 직접 바꿔 볼 손잡이

| 손잡이 | 위치 | 기본값 | 바꾸면 |
|---|---|---|---|
| `HH`, `N_B` | 2절 | 32, 4 | 공간 크기·배치. 키우면 간극이 어떻게 변하는가 |
| `CS` | 2절 | …256 | 채널 훑기 범위 |
| `M`, `m` | 4절 | 1024, 64 | 큰 곱과 작은 곱의 입도 |
| `rep` | 1절 | 5 | 측정 반복 수 (최솟값 채택) |

In [ ]:
print(f"총 실행 시간: {time.time() - _t0:.1f}초")